# 🎬 Ai-mation — توليد فيديو مجاني على Google Colab (من الهاتف)

يعمل بنموذج **LTX-Video 0.9.5** المفتوح على كرت شاشة **T4 المجاني** من Google.

## الخطوات
1. من القائمة: **Runtime ← Change runtime type ← T4 GPU ← Save**
   (على الهاتف: اضغط ⋮ أو اسم الملف ← Runtime)
2. شغّل الخلايا **بالترتيب** (1️⃣ ثم 2️⃣ ثم 3️⃣) بالضغط على ▶️ بجانب كل خلية.
3. الخلية 1 و2 تأخذ 5-10 دقائق في كل جلسة جديدة (تحميل النموذج ~25GB على سيرفر Google، وليس على هاتفك). بعدها كل فيديو يأخذ دقائق قليلة.
4. **🎞️ فيديو طويل**: عدة مشاهد متصلة في فيديو واحد.
5. **🎙️ تعليق صوتي**: صوت عربي (عراقي، سعودي، مصري، إماراتي) + ترجمة مكتوبة على الفيديو.
6. غيّر الوصف في الخلية 3 واضغط ▶️ لتوليد فيديو جديد — بدون إعادة الخلايا 1 و2.

> ⚠️ الجلسة المجانية محدودة بساعات يومياً، وقد لا يتوفر T4 أوقات الذروة. إذا أُغلقت الجلسة أعد تشغيل الخلايا من البداية.

## 💡 سر النتيجة الجيدة
- **وصف طويل ومفصل**: الشكل، الملابس، المكان، الحركة بالضبط، الإضاءة.
- **حركة واحدة بسيطة** في كل مشهد (تمشي، تلتفت، تبتسم) بدل أحداث كثيرة.
- **الكاميرا الثابتة** تعطي أقل تشوه.
- إذا أعجبك مشهد احفظ رقم **seed** وأعد استخدامه.


In [ ]:
#@title 1️⃣ تجهيز البيئة (مرة واحدة)
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ لا يوجد GPU. اذهب إلى Runtime ← Change runtime type ← اختر T4 GPU ثم أعد التشغيل.")
print("✅ GPU:", gpu)
# نسخة torchao المثبتة مسبقاً في Colab لا تتوافق مع diffusers، ولا نحتاجها
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 "bitsandbytes>=0.48.0" accelerate sentencepiece imageio imageio-ffmpeg pillow-heif
print("✅ تم التثبيت")

In [ ]:
#@title 2️⃣ تحميل النموذج (مرة واحدة لكل جلسة)
#@markdown إذا خرج الفيديو أسود أو مشوّه، غيّر الدقة إلى bfloat16 وأعد تشغيل هذه الخلية.
precision = "float16" #@param ["float16", "bfloat16"]

import torch, gc
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import LTXConditionPipeline

# الإصدار 0.9.5: حركة أكثر ثباتاً وأفضل بكثير في تحريك الصور وإكمال المشاهد
MODEL = "Lightricks/LTX-Video-0.9.5"
# نسخة bf16 من نفس مُرمّز النص (T5-XXL): 9.5GB بدل 19GB
TEXT_ENCODER = "city96/t5-v1_1-xxl-encoder-bf16"
dtype = torch.float16 if precision == "float16" else torch.bfloat16

# إذا بقيت ذاكرة الكرت محجوزة من محاولة سابقة فاشلة، يجب إعادة تشغيل الجلسة
free, total = torch.cuda.mem_get_info()
if total - free > 2e9:
    raise SystemExit(
        f"⚠️ ذاكرة الكرت مشغولة ({(total - free) / 1e9:.1f} GB) من محاولة سابقة.\n"
        "اضغط Runtime ← Restart session، ثم شغّل الخلية 1️⃣ ثم هذه الخلية من جديد."
    )

print("⏳ (1/2) تحميل مُرمّز النص... قد يستغرق عدة دقائق")
# 4-bit يقلل استهلاك ذاكرة الكرت إلى ~3GB، وbfloat16 يمنع أخطاء T5 التي تحدث مع float16
text_encoder = T5EncoderModel.from_pretrained(
    TEXT_ENCODER,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    ),
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
gc.collect(); torch.cuda.empty_cache()

print("⏳ (2/2) تحميل نموذج الفيديو...")
# خط معالجة واحد للنص والصورة وإكمال الفيديو
pipe = LTXConditionPipeline.from_pretrained(MODEL, text_encoder=text_encoder, torch_dtype=dtype, low_cpu_mem_usage=True)
pipe.transformer.to("cuda")
pipe.vae.to("cuda")
pipe.vae.enable_tiling()
i2v = pipe
gc.collect(); torch.cuda.empty_cache()
# ---------- أدوات التوليد ----------
import os, re, time, random, urllib.parse, urllib.request, json
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs("/content/videos", exist_ok=True)
SIZES = {"طولي 9:16 (تيك توك/ريلز)": (768, 512), "عرضي 16:9 (يوتيوب)": (512, 768), "مربع 1:1": (640, 640)}
NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed, morphing, warping, flickering, chaotic movement, extra limbs, watermark, text, nsfw, nudity"
# يضاف لكل وصف: النموذج يتحرك عشوائياً إذا كان الوصف قصيراً ولا يحدد الكاميرا
CAMERA = {
    "ثابتة": "The camera remains stationary.",
    "تقترب ببطء": "The camera slowly pushes in.",
    "تبتعد ببطء": "The camera slowly pulls back.",
    "تتبع الشخصية": "The camera smoothly follows the subject.",
    "تتحرك أفقياً": "The camera pans slowly from left to right.",
}
QUALITY = "Smooth, natural and consistent motion. The scene is captured in real-life footage, highly detailed."

def enrich(prompt_en, camera="ثابتة"):
    return f"{prompt_en.rstrip('. ')}. {CAMERA.get(camera, '')} {QUALITY}"
BLOCKED = ["nude", "naked", "nsfw", "porn", "sex", "erotic", "hentai", "topless",
           "عاري", "عارية", "جنس", "جنسي", "إباحي", "اباحي", "سكس", "بدون ملابس"]

def translate(text):
    if not re.search(r"[\u0600-\u06FF]", text):
        return text
    try:
        url = "https://api.mymemory.translated.net/get?langpair=ar|en&q=" + urllib.parse.quote(text[:500])
        t = json.load(urllib.request.urlopen(url, timeout=20))["responseData"]["translatedText"]
        print("🌐 الترجمة:", t)
        return t
    except Exception as e:
        print("⚠️ تعذرت الترجمة، سيُستخدم النص كما هو:", e)
        return text

def check(text):
    low = " " + re.sub(r"[^\w]+", " ", text.lower()) + " "
    if any(f" {w} " in low for w in BLOCKED):
        raise SystemExit("❌ الوصف يحتوي محتوى جنسياً صريحاً، وهذا غير مدعوم.")

def frames_for(seconds):
    n = int(seconds * 24)
    return (n // 8) * 8 + 1  # النموذج يحتاج عدد إطارات = 8k+1

def show(path):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="100%" controls autoplay loop playsinline src="data:video/mp4;base64,{data}"></video>'))
    print("💾 محفوظ في:", path, "— لتحميله: افتح أيقونة 📁 على اليسار، ثم ⋮ بجانب الملف ← Download")

def generate(pipeline, prompt_en, h, w, seconds, steps, seed, **extra):
    """يولّد مشهداً واحداً ويرجع قائمة الإطارات"""
    # نرمّز النص بأنفسنا ونحوّله إلى دقة نموذج الفيديو
    with torch.no_grad():
        pe, pm, ne, nm = pipeline.encode_prompt(
            prompt=prompt_en, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
            max_sequence_length=256, device="cuda", dtype=dtype,
        )
    frames = pipeline(
        prompt_embeds=pe, prompt_attention_mask=pm,
        negative_prompt_embeds=ne, negative_prompt_attention_mask=nm,
        width=w, height=h, num_frames=frames_for(seconds), frame_rate=24,
        num_inference_steps=steps, guidance_scale=3.0,
        decode_timestep=0.05, decode_noise_scale=0.025,
        generator=torch.Generator("cuda").manual_seed(seed), **extra,
    ).frames[0]
    gc.collect(); torch.cuda.empty_cache()
    return frames

def save_frames(frames, name="aimation"):
    path = f"/content/videos/{name}_{int(time.time())}.mp4"
    export_to_video(frames, path, fps=24)
    return path

def run(pipeline, prompt, size, seconds, steps, seed, camera="ثابتة", **extra):
    check(prompt)
    prompt_en = enrich(translate(prompt), camera)
    check(prompt_en)
    h, w = SIZES[size]
    seed = seed if seed >= 0 else random.randint(0, 2**31 - 1)
    print(f"🎬 جارٍ التوليد... ({seconds} ثوانٍ، {steps} خطوة، seed={seed})")
    t0 = time.time()
    frames = generate(pipeline, prompt_en, h, w, seconds, steps, seed, **extra)
    path = save_frames(frames)
    print(f"✅ تم في {time.time() - t0:.0f} ثانية")
    show(path)
    return path

print("✅ جاهز")

print(f"✅ النموذج جاهز — ذاكرة الكرت المستخدمة: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
#@title 3️⃣ نص ← فيديو
#@markdown اكتب وصفاً **مفصلاً**: من الشخصية وشكلها وملابسها، أين هي، ماذا تفعل بالضبط، الإضاءة. الوصف القصير = حركة عشوائية.
prompt = "محارب صحراوي يمشي ببطء أمام قلعة قديمة عند الغروب، الرياح تحرك عباءته، الكاميرا تتبعه بهدوء، إضاءة سينمائية" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
#@markdown ↑ الكاميرا الثابتة تعطي أقل تشوه
#@markdown seed = -1 يعني عشوائي. استخدم نفس الرقم لتكرار نفس النتيجة.

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")
video_path = run(pipe, prompt, size, seconds, steps, seed, camera)

In [ ]:
#@title 🖼️ اختيار صورة للتحريك
#@markdown شغّل هذه الخلية (تعمل فوراً)، ثم اضغط زر **🖼️ اختر صورة من الهاتف**. بعد ظهور ✅ شغّل خلية 4️⃣.
import base64, os, time
from google.colab import output
from IPython.display import HTML, display

MIME_EXT = {"image/jpeg": ".jpg", "image/png": ".png", "image/webp": ".webp", "image/heic": ".heic", "image/heif": ".heic"}

def _save_upload(name, data_url):
    header, b64 = data_url.split(",", 1)
    mime = header[5:].split(";")[0]
    ext = MIME_EXT.get(mime) or (os.path.splitext(name)[1].lower() or ".jpg")
    path = f"/content/upload_{int(time.time())}{ext}"
    with open(path, "wb") as f:
        f.write(base64.b64decode(b64))
    return path

output.register_callback("aimation.saveImage", _save_upload)
display(HTML('<div dir="rtl" style="font-family:sans-serif;padding:8px 0">\n  <label for="aimFile" style="display:inline-block;padding:18px 28px;font-size:20px;font-weight:bold;color:#fff;border-radius:14px;cursor:pointer;background:linear-gradient(90deg,#8b5cf6,#ec4899)">🖼️ اختر صورة من الهاتف</label>\n  <input id="aimFile" type="file" accept="image/*" style="display:none">\n  <div id="aimStatus" style="margin-top:12px;font-size:16px"></div>\n  <img id="aimPreview" style="display:none;margin-top:10px;max-width:60%;max-height:360px;border-radius:10px">\n</div>\n<script>\n(() => {\n  const input = document.getElementById("aimFile");\n  const status = document.getElementById("aimStatus");\n  const preview = document.getElementById("aimPreview");\n  const readAsDataURL = (file) => new Promise((ok, fail) => {\n    const r = new FileReader(); r.onload = () => ok(r.result); r.onerror = fail; r.readAsDataURL(file);\n  });\n  // نصغّر الصورة قبل الرفع (أسرع وأخف)، وإذا لم يستطع المتصفح فتحها (مثل HEIC) نرسلها كما هي\n  const shrink = (dataUrl) => new Promise((ok) => {\n    const img = new Image();\n    img.onload = () => {\n      const k = Math.min(1, 1536 / Math.max(img.width, img.height));\n      const c = document.createElement("canvas");\n      c.width = Math.round(img.width * k); c.height = Math.round(img.height * k);\n      c.getContext("2d").drawImage(img, 0, 0, c.width, c.height);\n      ok({ url: c.toDataURL("image/jpeg", 0.92), shown: true });\n    };\n    img.onerror = () => ok({ url: dataUrl, shown: false });\n    img.src = dataUrl;\n  });\n  input.onchange = async () => {\n    const file = input.files[0];\n    if (!file) return;\n    status.textContent = "⏳ جارٍ رفع الصورة...";\n    try {\n      const raw = await readAsDataURL(file);\n      const { url, shown } = await shrink(raw);\n      if (shown) { preview.src = url; preview.style.display = "block"; }\n      await google.colab.kernel.invokeFunction("aimation.saveImage", [file.name, url], {});\n      status.textContent = "✅ تم رفع الصورة. الآن شغّل خلية 4️⃣ صورة ← فيديو.";\n    } catch (e) {\n      status.textContent = "❌ فشل الرفع: " + e + " — تأكد أن الجلسة متصلة ثم أعد الاختيار.";\n    }\n    input.value = "";\n  };\n})();\n</script>\n'))


In [ ]:
#@title 4️⃣ صورة ← فيديو (تحريك صورة)
#@markdown **أولاً:** ارفع الصورة من خلية **🖼️ اختيار صورة للتحريك** التي فوق هذه الخلية. ستُستخدم أحدث صورة رفعتها تلقائياً.
#@markdown (أو الصق رابط صورة في خانة image)
motion = "الشخصية تبتسم وتلتفت ببطء نحو الكاميرا، الشعر يتحرك مع النسيم" #@param {type:"string"}
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
seed = -1 #@param {type:"integer"}
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
#@markdown ↑ الكاميرا الثابتة تعطي أقل تشوه
image = "" #@param {type:"string"}
#@markdown ↑ اتركه فارغاً لاستخدام أحدث صورة مرفوعة، أو اكتب اسم الملف، أو الصق رابط صورة (https://...)

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")

import glob, io
from PIL import Image, ImageOps
try:  # صور HEIC من بعض الهواتف
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    pass

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic")
src = image.strip()
if src.startswith("http"):
    print("🌐 تحميل الصورة من الرابط...")
    req = urllib.request.Request(src, headers={"User-Agent": "Mozilla/5.0"})
    img = Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=30).read()))
    name = src
else:
    if src:
        candidates = [src if src.startswith("/") else f"/content/{src}"]
    else:
        candidates = sorted(
            (f for f in glob.glob("/content/*") + glob.glob("/content/drive/MyDrive/*")
             if f.lower().endswith(IMG_EXT)),
            key=os.path.getmtime,
        )[-1:]
    if not candidates or not os.path.exists(candidates[0]):
        raise SystemExit(
            "❌ لم أجد صورة.\n"
            "ارفعها من: ☰ ← Show file browser ← أيقونة الرفع ⬆️، وانتظر ظهورها في القائمة، ثم أعد تشغيل الخلية.\n"
            "أو الصق رابط صورة في خانة image."
        )
    name = candidates[0]
    img = Image.open(name)
print("🖼️ الصورة:", os.path.basename(name))

h, w = SIZES[size]
img = ImageOps.fit(ImageOps.exif_transpose(img).convert("RGB"), (w, h))
display(img.resize((w // 3, h // 3)))
video_path = run(i2v, motion, size, seconds, steps, seed, camera, image=img)


In [ ]:
#@title 🎞️ فيديو طويل (عدة مشاهد متصلة)
#@markdown اكتب المشاهد مفصولة بعلامة **|** . اكتب في خانة **character** وصف الشخصية والمكان مرة واحدة ليُضاف لكل مشهد فتبقى متطابقة.
scenes = "فتاة تمشي في سوق شعبي قديم نهاراً | الفتاة تتوقف أمام بائع التوابل وتبتسم | الكاميرا ترتفع ببطء لتظهر السوق كله من الأعلى" #@param {type:"string"}
character = "فتاة شابة بشعر أسود طويل ترتدي فستاناً أحمر، في سوق شعبي مزدحم بالبضائع الملونة" #@param {type:"string"}
style = "سينمائي، إضاءة دافئة" #@param {type:"string"}
#@markdown ↑ نمط يُضاف لكل المشاهد ليبقى الشكل موحداً (اختياري)
size = "طولي 9:16 (تيك توك/ريلز)" #@param ["طولي 9:16 (تيك توك/ريلز)", "عرضي 16:9 (يوتيوب)", "مربع 1:1"]
seconds_per_scene = 4 #@param {type:"slider", min:2, max:6, step:1}
steps = 30 #@param {type:"slider", min:15, max:50, step:5}
transition = "مونتاج (قطع بين المشاهد)" #@param ["مونتاج (قطع بين المشاهد)", "متصل (إكمال من آخر المشهد)"]
#@markdown ↑ **المونتاج** أوضح وأقل تشوهاً (مثل الأفلام). **المتصل** حركة واحدة مستمرة لكن قد تتراكم الأخطاء.
camera = "ثابتة" #@param ["ثابتة", "تقترب ببطء", "تبتعد ببطء", "تتبع الشخصية", "تتحرك أفقياً"]
seed = -1 #@param {type:"integer"}

if "run" not in globals() or "pipe" not in globals():
    raise SystemExit("⚠️ شغّل الخلية 2️⃣ أولاً وانتظر رسالة ✅ النموذج جاهز، ثم شغّل هذه الخلية.")

parts = [x.strip() for x in scenes.split("|") if x.strip()]
if not parts:
    raise SystemExit("اكتب مشهداً واحداً على الأقل.")
check(scenes + " " + style + " " + character)
style_en = translate(style) if style.strip() else ""
char_en = translate(character) if character.strip() else ""
h, w = SIZES[size]
seed = seed if seed >= 0 else random.randint(0, 2**31 - 1)
print(f"🎞️ {len(parts)} مشاهد × {seconds_per_scene} ثوانٍ ≈ {len(parts) * seconds_per_scene} ثانية (seed={seed})")

all_frames, t0 = [], time.time()
for i, scene in enumerate(parts):
    prompt_en = enrich(". ".join(x for x in [char_en, translate(scene), style_en] if x), camera)
    check(prompt_en)
    print(f"\n▶️ المشهد {i + 1}/{len(parts)}: {prompt_en}")
    if i > 0 and transition.startswith("متصل"):
        # نكمل من آخر 9 إطارات (وليس صورة واحدة) لتستمر الحركة بسلاسة
        tail = all_frames[-9:]
        frames = generate(pipe, prompt_en, h, w, seconds_per_scene, steps, seed + i, video=tail)
        frames = frames[len(tail):]  # بداية المقطع الجديد مكررة من المشهد السابق
    else:
        frames = generate(pipe, prompt_en, h, w, seconds_per_scene, steps, seed + i)
    all_frames.extend(frames)

video_path = save_frames(all_frames, "aimation_long")
print(f"\n✅ تم في {(time.time() - t0) / 60:.1f} دقيقة")
show(video_path)

In [ ]:
#@title 🙂 إصلاح الوجوه
#@markdown يكتشف الوجوه في كل إطار ويصلحها بنموذج GFPGAN. يعمل على آخر فيديو، أو اكتب اسم ملف.
video_file = "" #@param {type:"string"}
strength = 0.7 #@param {type:"slider", min:0.3, max:1.0, step:0.1}
#@markdown ↑ قوة الإصلاح: 0.7 متوازن، 1.0 أقصى إصلاح (قد يبدو الوجه مصقولاً أكثر من اللازم)

!pip install -q onnxruntime-gpu opencv-python-headless
import os, glob, time, subprocess, json
import cv2
import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download

if "show" not in globals():
    from IPython.display import HTML, display
    from base64 import b64encode
    def show(path):
        data = b64encode(open(path, "rb").read()).decode()
        display(HTML(f'<video width="100%" controls playsinline src="data:video/mp4;base64,{data}"></video>'))
        print("💾 محفوظ في:", path)


# قالب مواضع العينين والأنف والفم في صورة وجه 512×512 (FFHQ)
FACE_TEMPLATE = np.array([
    [0.37691676, 0.46864664], [0.62285697, 0.46912813], [0.50123859, 0.61331904],
    [0.39308822, 0.72541100], [0.61150205, 0.72490465],
], dtype=np.float32) * 512


def detect_faces(det, frame, min_size=40, threshold=0.5):
    """YOLOv8-face: يرجع [(box, landmarks5)]"""
    h, w = frame.shape[:2]
    scale = min(640 / w, 640 / h)
    rw, rh = int(w * scale), int(h * scale)
    canvas = np.zeros((640, 640, 3), np.float32)
    canvas[:rh, :rw] = cv2.resize(frame, (rw, rh))[:, :, ::-1]
    x = (canvas / 255.0).transpose(2, 0, 1)[None].astype(np.float32)
    out = np.squeeze(det.run(None, {det.get_inputs()[0].name: x})[0]).T
    boxes, scores, lms = out[:, :4], out[:, 4], out[:, 5:]
    keep = scores > threshold
    boxes, scores, lms = boxes[keep] / scale, scores[keep], lms[keep]
    if not len(boxes):
        return []
    xywh = np.stack([boxes[:, 0] - boxes[:, 2] / 2, boxes[:, 1] - boxes[:, 3] / 2, boxes[:, 2], boxes[:, 3]], 1)
    idx = cv2.dnn.NMSBoxes(xywh.tolist(), scores.tolist(), threshold, 0.4)
    faces = []
    for i in np.array(idx).flatten():
        if xywh[i, 2] < min_size:
            continue  # الوجوه الصغيرة جداً يفسدها الإصلاح
        lm = lms[i].reshape(5, 3)[:, :2] / scale
        faces.append((xywh[i], lm.astype(np.float32)))
    return faces


def box_mask(size=512, blur=0.3):
    amount = int(size * 0.5 * blur)
    area = max(amount // 2, 1)
    mask = np.ones((size, size), np.float32)
    mask[:area, :] = mask[-area:, :] = 0
    mask[:, :area] = mask[:, -area:] = 0
    return cv2.GaussianBlur(mask, (0, 0), amount * 0.25) if amount > 0 else mask


MASK = box_mask()


def restore_face(gfp, frame, lm, strength):
    h, w = frame.shape[:2]
    M = cv2.estimateAffinePartial2D(lm, FACE_TEMPLATE, method=cv2.RANSAC, ransacReprojThreshold=100)[0]
    if M is None:
        return frame
    crop = cv2.warpAffine(frame, M, (512, 512), borderMode=cv2.BORDER_REPLICATE, flags=cv2.INTER_AREA)
    x = ((crop[:, :, ::-1] / 255.0 - 0.5) / 0.5).transpose(2, 0, 1)[None].astype(np.float32)
    y = gfp.run(None, {gfp.get_inputs()[0].name: x})[0][0]
    y = ((np.clip(y, -1, 1) + 1) / 2 * 255).transpose(1, 2, 0).round().astype(np.uint8)[:, :, ::-1]
    y = cv2.addWeighted(crop, 1 - strength, y, strength, 0)
    inv = cv2.invertAffineTransform(M)
    m = cv2.warpAffine(MASK, inv, (w, h)).clip(0, 1)[:, :, None]
    face = cv2.warpAffine(y, inv, (w, h), borderMode=cv2.BORDER_REPLICATE)
    return (m * face + (1 - m) * frame).astype(np.uint8)


class Smoother:
    """يثبّت نقاط الوجه بين الإطارات لتقليل الارتجاف"""

    def __init__(self, alpha=0.5):
        self.alpha, self.prev = alpha, []

    def __call__(self, faces):
        out = []
        for box, lm in faces:
            c = lm.mean(0)
            near = [p for p in self.prev if np.linalg.norm(p.mean(0) - c) < box[2] * 0.3]
            if near:
                lm = self.alpha * lm + (1 - self.alpha) * near[0]
            out.append((box, lm))
        self.prev = [lm for _, lm in out]
        return out


def process_video(det, gfp, src, dst, strength=0.7, progress=None):
    cap = cv2.VideoCapture(src)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    smooth, n, fixed = Smoother(), 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        faces = smooth(detect_faces(det, frame))
        for _, lm in faces:
            frame = restore_face(gfp, frame, lm, strength)
        fixed += bool(faces)
        out.write(frame)
        n += 1
        if progress and n % 12 == 0:
            progress(n, total)
    cap.release()
    out.release()
    return n, fixed


def _latest_video():
    vids = [v for v in glob.glob("/content/videos/*.mp4") + glob.glob("/content/*.mp4")
            if not any(t in v for t in ("_faces", "_voice", "montage"))]
    return max(vids, key=os.path.getmtime) if vids else None

src = video_file.strip()
src = (src if src.startswith("/") else f"/content/videos/{src}") if src else _latest_video()
if not src or not os.path.exists(src):
    raise SystemExit("❌ لا يوجد فيديو. ولّد فيديو أولاً، أو اكتب اسم الملف في خانة video_file.")
print("🎬 الفيديو:", os.path.basename(src))

if "FACE_MODELS" not in globals():
    print("⏳ تحميل نماذج الوجوه (مرة واحدة)...")
    if hasattr(ort, "preload_dlls"):
        try:
            ort.preload_dlls()  # مكتبات CUDA المثبتة مع PyTorch
        except Exception:
            pass
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    repo = "facefusion/models-3.0.0"
    FACE_MODELS = (
        ort.InferenceSession(hf_hub_download(repo, "yoloface_8n.onnx"), providers=providers),
        ort.InferenceSession(hf_hub_download(repo, "gfpgan_1.4.onnx"), providers=providers),
    )
    print("⚙️ يعمل على:", "GPU ⚡" if "CUDAExecutionProvider" in FACE_MODELS[1].get_providers() else "CPU (أبطأ)")
det, gfp = FACE_MODELS

tmp = src[:-4] + "_tmp.mp4"
out = src[:-4] + "_faces.mp4"
t0 = time.time()
n, fixed = process_video(det, gfp, src, tmp, strength,
                         lambda i, total: print(f"\r🙂 {i}/{total} إطار", end=""))
print()
# ترميز نهائي متوافق مع الهاتف، مع الإبقاء على الصوت إن وجد
r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp, "-i", src, "-map", "0:v", "-map", "1:a?",
                    "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "copy", out],
                   capture_output=True, text=True)
os.remove(tmp)
if r.returncode != 0:
    raise SystemExit("❌ فشل الترميز:\n" + r.stderr[-1000:])
video_path = out
print(f"✅ تم إصلاح الوجوه في {fixed} من {n} إطار خلال {time.time() - t0:.0f} ثانية")
if fixed == 0:
    print("ℹ️ لم يُكتشف أي وجه واضح (الوجوه الصغيرة جداً أو الجانبية تُتجاهل).")
show(out)

In [ ]:
#@title ✂️ مونتاج آلي
#@markdown يجمع مقاطعك في فيديو واحد: يأخذ أفضل جزء (البداية) من كل مقطع، يوحّد الألوان، يضيف انتقالات وتكبيراً بطيئاً، ويقطع على إيقاع الموسيقى.
#@markdown **الموسيقى:** ارفع ملف mp3 من ☰ ← Show file browser ← ⬆️ وسيُستخدم أحدث ملف صوتي تلقائياً.
clips = "" #@param {type:"string"}
#@markdown ↑ اتركه فارغاً لاستخدام كل المقاطع بترتيب توليدها، أو اكتب أسماء الملفات مفصولة بفاصلة بالترتيب الذي تريده
clip_seconds = 3.0 #@param {type:"slider", min:1.5, max:6, step:0.5}
#@markdown ↑ مدة كل لقطة (المونتاج الاحترافي 2-3 ثوانٍ)
transition = "تلاشي ناعم" #@param ["تلاشي ناعم", "قطع مباشر", "تلاشي للأسود", "انزلاق"]
color = "دافئ ذهبي" #@param ["دافئ ذهبي", "سينمائي", "تراثي قديم", "بارد", "طبيعي"]
slow_zoom = True #@param {type:"boolean"}
resolution = "طولي 9:16" #@param ["طولي 9:16", "عرضي 16:9", "مربع 1:1"]
use_music = True #@param {type:"boolean"}
beat_sync = True #@param {type:"boolean"}
music_volume = 0.8 #@param {type:"slider", min:0.1, max:1.0, step:0.1}

import os, glob, time, subprocess, json

if "show" not in globals():
    from IPython.display import HTML, display
    from base64 import b64encode
    def show(path):
        data = b64encode(open(path, "rb").read()).decode()
        display(HTML(f'<video width="100%" controls playsinline src="data:video/mp4;base64,{data}"></video>'))
        print("💾 محفوظ في:", path)


COLORS = {
    "دافئ ذهبي": "colorbalance=rs=0.06:gs=0.02:bs=-0.07:rm=0.04:bm=-0.04,eq=saturation=1.1:contrast=1.05",
    "سينمائي": "colorbalance=rs=-0.05:bs=0.06:rh=0.06:bh=-0.06,eq=contrast=1.1:saturation=0.9,vignette=PI/5",
    "تراثي قديم": "colorchannelmixer=.393:.769:.189:0:.349:.686:.168:0:.272:.534:.131,eq=contrast=1.05,vignette=PI/4",
    "بارد": "colorbalance=rs=-0.05:bs=0.07,eq=saturation=0.95",
    "طبيعي": "",
}
RESOLUTIONS = {"طولي 9:16": (720, 1280), "عرضي 16:9": (1280, 720), "مربع 1:1": (1080, 1080)}
FPS = 24


def duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", path],
                         capture_output=True, text=True).stdout
    return float(json.loads(out)["format"]["duration"])


def beat_period(music):
    import librosa
    y, sr = librosa.load(music, sr=22050, duration=120)
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(tempo[0] if hasattr(tempo, "__len__") else tempo)
    first = float(librosa.frames_to_time(beats[0], sr=sr)) if len(beats) else 0.0
    return (60.0 / tempo if tempo > 0 else None), first, tempo


def plan_lengths(durations, clip_len, period=None):
    """طول كل مقطع: الجزء الأول (الأنظف) منه، ومضاعف للإيقاع إن وجدت موسيقى"""
    out = []
    for d in durations:
        L = min(clip_len, d)
        if period:
            beats = max(1, int(round(L / period)))
            while beats > 1 and beats * period > d:
                beats -= 1
            L = min(beats * period, d)
        out.append(round(L, 3))
    return out


def build_cmd(clips, lengths, out, size, color, kenburns, transition, T, music=None,
              music_start=0.0, music_vol=0.8):
    W, H = size
    n = len(clips)
    T = T if transition != "قطع مباشر" and n > 1 else 0
    T = min([T] + [L / 2 for L in lengths])
    total = sum(lengths) - T * (n - 1)
    cmd = ["ffmpeg", "-y", "-loglevel", "error"]
    for c in clips:
        cmd += ["-i", c]
    if music:
        cmd += ["-stream_loop", "-1", "-ss", f"{music_start:.3f}", "-i", music]
    parts = []
    for i, L in enumerate(lengths):
        f = [f"[{i}:v]trim=0:{L}", "setpts=PTS-STARTPTS", f"fps={FPS}",
             f"scale={W}:{H}:force_original_aspect_ratio=increase:flags=lanczos", f"crop={W}:{H}", "setsar=1"]
        if COLORS.get(color):
            f.append(COLORS[color])
        if kenburns:
            frames = max(int(L * FPS), 1)
            step = 0.10 / frames
            z = f"min(1+{step:.6f}*on,1.10)" if i % 2 == 0 else f"max(1.10-{step:.6f}*on,1)"
            f.append(f"zoompan=z='{z}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d=1:s={W}x{H}:fps={FPS}")
        f.append("format=yuv420p")
        parts.append(",".join(f) + f"[v{i}]")
    if n == 1:
        last = "v0"
    elif T > 0:
        kind = {"تلاشي ناعم": "fade", "تلاشي للأسود": "fadeblack", "انزلاق": "slideleft"}.get(transition, "fade")
        last, offset = "v0", 0.0
        for i in range(1, n):
            offset += lengths[i - 1] - T
            parts.append(f"[{last}][v{i}]xfade=transition={kind}:duration={T}:offset={offset:.3f}[x{i}]")
            last = f"x{i}"
    else:
        parts.append("".join(f"[v{i}]" for i in range(n)) + f"concat=n={n}:v=1:a=0[cat]")
        last = "cat"
    parts.append(f"[{last}]fade=t=in:d=0.4,fade=t=out:st={max(total - 0.6, 0):.3f}:d=0.6[vout]")
    maps = ["-map", "[vout]"]
    if music:
        parts.append(f"[{n}:a]atrim=0:{total:.3f},asetpts=PTS-STARTPTS,afade=t=in:d=0.8,"
                     f"afade=t=out:st={max(total - 1.5, 0):.3f}:d=1.5,volume={music_vol}[aout]")
        maps += ["-map", "[aout]", "-c:a", "aac", "-b:a", "192k"]
    cmd += ["-filter_complex", ";".join(parts), *maps,
            "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
            "-movflags", "+faststart", "-t", f"{total:.3f}", out]
    return cmd, total


def _pick_clips():
    if clips.strip():
        names = [c.strip() for c in clips.split(",") if c.strip()]
        paths = []
        for nme in names:
            cand = [nme] if nme.startswith("/") else [f"/content/videos/{nme}", f"/content/{nme}"]
            found = next((c for c in cand if os.path.exists(c)), None)
            if not found:
                raise SystemExit(f"❌ لم أجد الملف: {nme}")
            paths.append(found)
        return paths
    vids = [v for v in glob.glob("/content/videos/*.mp4") + glob.glob("/content/*.mp4")
            if not any(t in v for t in ("_voice", "montage", "_tmp"))]
    # إذا وُجدت نسخة بوجوه مُصلحة نستخدمها بدل الأصلية
    fixed = {v.replace("_faces.mp4", ".mp4") for v in vids if v.endswith("_faces.mp4")}
    vids = [v for v in vids if v not in fixed]
    return sorted(vids, key=os.path.getmtime)

files = _pick_clips()
if not files:
    raise SystemExit("❌ لا توجد مقاطع. ولّد مقاطع أولاً أو ارفعها إلى /content.")
print(f"🎞️ {len(files)} مقاطع:", ", ".join(os.path.basename(f) for f in files))

music, period, start = None, None, 0.0
if use_music:
    audio = [a for a in glob.glob("/content/*") if a.lower().endswith((".mp3", ".m4a", ".wav", ".aac", ".ogg"))]
    if audio:
        music = max(audio, key=os.path.getmtime)
        print("🎵 الموسيقى:", os.path.basename(music))
        if beat_sync:
            try:
                period, start, tempo = beat_period(music)
                print(f"🥁 الإيقاع: {tempo:.0f} BPM — القطع على كل {period:.2f} ثانية")
            except Exception as e:
                print("⚠️ تعذر تحليل الإيقاع، سيتم القطع بدون مزامنة:", e)
    else:
        print("ℹ️ لا يوجد ملف موسيقى في /content، سيكون الفيديو بدون موسيقى.")

lengths = plan_lengths([duration(f) for f in files], clip_seconds, period)
out = f"/content/videos/aimation_montage_{int(time.time())}.mp4"
cmd, total = build_cmd(files, lengths, out, RESOLUTIONS[resolution], color, slow_zoom,
                       transition, 0.5, music, start, music_volume)
print(f"⏳ جارٍ المونتاج... (المدة النهائية ≈ {total:.1f} ثانية)")
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ فشل المونتاج:\n" + r.stderr[-1500:])
video_path = out
print(f"✅ تم في {time.time() - t0:.0f} ثانية")
show(out)

In [ ]:
#@title 🎙️ تعليق صوتي عربي + ترجمة على الفيديو
#@markdown يضيف صوتاً عربياً طبيعياً لآخر فيديو (من أي خلية، ومنها المونتاج). إذا كان في الفيديو موسيقى تُخفض تلقائياً تحت الصوت.
narration = "في قلب المدينة القديمة، حيث تحكي الجدران قصص الأجداد… بدأت الحكاية." #@param {type:"string"}
voice = "عراقي - رجل (باسل)" #@param ["عراقي - رجل (باسل)", "عراقي - امرأة (رنا)", "سعودي - رجل (حامد)", "سعودية - امرأة (زارية)", "مصري - رجل (شاكر)", "مصرية - امرأة (سلمى)", "إماراتي - رجل (حمدان)", "إماراتية - امرأة (فاطمة)"]
speed = 0 #@param {type:"slider", min:-30, max:30, step:5}
#@markdown ↑ سرعة الكلام بالنسبة المئوية (0 = عادي)
subtitles = True #@param {type:"boolean"}
video_file = "" #@param {type:"string"}
#@markdown ↑ اتركه فارغاً لاستخدام آخر فيديو، أو اكتب اسم ملف من مجلد videos

!pip install -q edge-tts==7.3.1
import edge_tts, glob, subprocess, json as _json

VOICES = {
    "عراقي - رجل (باسل)": "ar-IQ-BasselNeural", "عراقي - امرأة (رنا)": "ar-IQ-RanaNeural",
    "سعودي - رجل (حامد)": "ar-SA-HamedNeural", "سعودية - امرأة (زارية)": "ar-SA-ZariyahNeural",
    "مصري - رجل (شاكر)": "ar-EG-ShakirNeural", "مصرية - امرأة (سلمى)": "ar-EG-SalmaNeural",
    "إماراتي - رجل (حمدان)": "ar-AE-HamdanNeural", "إماراتية - امرأة (فاطمة)": "ar-AE-FatimaNeural",
}

if not narration.strip():
    raise SystemExit("اكتب نص التعليق أولاً.")
check(narration)

if video_file.strip():
    src = video_file if video_file.startswith("/") else f"/content/videos/{video_file.strip()}"
else:
    vids = sorted((v for v in glob.glob("/content/videos/*.mp4") if "_voice" not in v), key=os.path.getmtime)
    if not vids:
        raise SystemExit("لا يوجد فيديو بعد. ولّد فيديو من الخلية 3️⃣ أو 4️⃣ أو 🎞️ أولاً.")
    src = vids[-1]
print("🎬 الفيديو:", os.path.basename(src))

# 1) توليد الصوت والترجمة
base = src[:-4]
audio, srt = base + "_voice.mp3", base + ".srt"
communicate = edge_tts.Communicate(narration, VOICES[voice], rate=f"{speed:+d}%")
submaker = edge_tts.SubMaker()
with open(audio, "wb") as f:
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            f.write(chunk["data"])
        elif chunk["type"] in ("WordBoundary", "SentenceBoundary"):
            submaker.feed(chunk)
open(srt, "w", encoding="utf-8").write(submaker.get_srt())
print("🎙️ تم توليد الصوت")

def duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", path],
                         capture_output=True, text=True).stdout
    return float(_json.loads(out)["format"]["duration"])

v_len, a_len = duration(src), duration(audio)
print(f"⏱️ الفيديو {v_len:.1f}ث — الصوت {a_len:.1f}ث")

# 2) خط عربي للترجمة
vf = []
if subtitles:
    font_dir = "/content/fonts"
    font = f"{font_dir}/NotoSansArabic.ttf"
    if not os.path.exists(font):
        os.makedirs(font_dir, exist_ok=True)
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/notosansarabic/NotoSansArabic%5Bwdth,wght%5D.ttf", font)
        except Exception as e:
            print("⚠️ تعذر تحميل الخط، سيتم الدمج بدون ترجمة:", e)
    if os.path.exists(font):
        sub_style = "FontName=Noto Sans Arabic,Bold=1,FontSize=13,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=0,MarginV=40,Alignment=2"
        vf = ["-vf", f"subtitles={srt}:fontsdir={font_dir}:force_style='{sub_style}'"]

# 3) الدمج
has_audio = bool(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index",
                                 "-of", "csv=p=0", src], capture_output=True, text=True).stdout.strip())
out = base + "_voice.mp4"
cmd = ["ffmpeg", "-y", "-loglevel", "error"]
if has_audio:
    # الفيديو فيه موسيقى (مثلاً من المونتاج): نخفض الموسيقى ونضع الصوت فوقها
    if a_len > v_len:
        print("⚠️ التعليق أطول من الفيديو وسيُقطع في النهاية. اختصر النص أو زد سرعة الكلام.")
    cmd += ["-i", src, "-i", audio,
            "-filter_complex", "[0:a]volume=0.25[m];[1:a]adelay=300|300[v];[m][v]amix=inputs=2:duration=first:dropout_transition=0:normalize=0[a]",
            *vf, "-map", "0:v", "-map", "[a]"]
else:
    if a_len > v_len:
        cmd += ["-stream_loop", "-1"]  # نكرر الفيديو حتى ينتهي الصوت
    cmd += ["-i", src, "-i", audio, *vf, "-map", "0:v", "-map", "1:a"]
    if a_len > v_len:
        cmd += ["-shortest"]
cmd += ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", out]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ فشل الدمج:\n" + r.stderr[-1500:])
video_path = out
print("✅ تم إضافة الصوت" + (" والترجمة" if vf else ""))
show(out)

In [ ]:
#@title 5️⃣ (اختياري) حفظ كل الفيديوهات في Google Drive
from google.colab import drive
import shutil
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Ai-mation"
os.makedirs(dest, exist_ok=True)
for f in os.listdir("/content/videos"):
    shutil.copy(f"/content/videos/{f}", dest)
print("✅ تم الحفظ في Google Drive ← مجلد Ai-mation")